### 評価値の算出

In [ ]:
import glob
import os
import pandas as pd


def process_csv_folder(target_dir: str):
    """指定フォルダ内の全CSVを読み込み、評価値(eval_value_pct)を追加・保存する関数"""
    # 前後の空白や引用符 (' や ") を自動除去
    target_dir = target_dir.strip().strip("'\"")

    if not os.path.exists(target_dir):
        print(f"エラー: 指定されたフォルダ '{target_dir}' が存在しません。")
        return

    csv_files = glob.glob(os.path.join(target_dir, "*.csv"))

    if not csv_files:
        print(
            f"指定されたフォルダ '{target_dir}' 内にCSVファイルが見つかりませんでした。"
        )
        return

    print(f"合計 {len(csv_files)} 件のCSVファイルを処理します...\n")

    for file_path in csv_files:
        df = pd.read_csv(file_path)

        # カラム名の自動識別
        comp_col = next(
            (c for c in ["comp_with_false", "sigma_comp"] if c in df.columns),
            None,
        )
        indep_col = next(
            (
                c
                for c in ["indep_with_false", "sigma_indep"]
                if c in df.columns
            ),
            None,
        )

        if comp_col and indep_col:
            # 評価式: (1 - (indep_with_false / comp_with_false)) * 100
            df["eval_value_pct"] = (1 - (df[indep_col] / df[comp_col])) * 100
            df["eval_value_pct"] = df["eval_value_pct"].round(2)

            df.to_csv(file_path, index=False)
            print(f"【更新成功】 {os.path.basename(file_path)}")
        else:
            print(
                f"【スキップ】 {os.path.basename(file_path)} : 対象の列名が存在しません。"
            )

    print("\nすべてのファイルの処理が完了しました。")

In [ ]:
target_folder = input(
    "../../data/LFR01/result/f_com1_user_2"
)
process_csv_folder(target_folder)

In [1]:
import glob
import os
import pandas as pd
from IPython.display import display


def evaluate_csv_folder(target_folder: str):
    """指定されたフォルダ内のCSVファイルに評価値(eval_value_pct)を追加・保存し、

    各ファイルの最高値と平均値を集計して表形式で出力する関数。

    引数:
        target_folder (str): CSVファイルが保存されているフォルダのパス
    """
    # パス前後の空白やクォーテーションの自動除去
    target_folder = target_folder.strip().strip("'\"")

    if not os.path.exists(target_folder):
        print(f"エラー: 指定されたフォルダ '{target_folder}' が存在しません。")
        return

    csv_files = glob.glob(os.path.join(target_folder, "*.csv"))

    if not csv_files:
        print(
            f"指定されたフォルダ '{target_folder}' 内にCSVファイルが見つかりませんでした。"
        )
        return

    print(f"合計 {len(csv_files)} 件のCSVファイルを処理中...\n")

    summary_data = []

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        df = pd.read_csv(file_path)

        # カラム名の自動識別
        comp_col = next(
            (
                c
                for c in ["comp_with_false", "sigma_comp", "ture_only"]
                if c in df.columns
            ),
            None,
        )
        indep_col = next(
            (
                c
                for c in ["indep_with_false", "sigma_indep", "ture_with_false"]
                if c in df.columns
            ),
            None,
        )

        if comp_col and indep_col:
            # 1. 評価値 (%) の計算と新規列の追加
            df["eval_value_pct"] = (
                1 - (df[indep_col] / df[comp_col])
            ) * 100
            df["eval_value_pct"] = df["eval_value_pct"].round(2)

            # 2. CSVファイルへ上書き保存
            df.to_csv(file_path, index=False)

            # 3. 各ファイルの最高値・平均値を算出
            max_val = df["eval_value_pct"].max()
            mean_val = df["eval_value_pct"].mean()

            summary_data.append({
                "ファイル名": file_name,
                "評価値の最高値 (%)": round(max_val, 2),
                "評価値の平均値 (%)": round(mean_val, 2),
            })
            print(f"【更新成功】 {file_name}")
        else:
            print(
                f"【スキップ】 {file_name} : 対象列が存在しません。"
            )

    # 4. 集計結果をPandasの表形式で出力
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        print("\n=================== 集計結果 ===================")
        display(summary_df)

In [2]:
# 相対パスで実行する場合
evaluate_csv_folder("../../data/LFR01/result/f_com1_user")

合計 7 件のCSVファイルを処理中...

【スキップ】 center_internal_pagerank_20261105_160846.csv : 対象列が存在しません。
【スキップ】 periphery_multiple_20261105_160846.csv : 対象列が存在しません。
【スキップ】 bridge_both_20261105_160846.csv : 対象列が存在しません。
【スキップ】 bridge_v_only_20261105_160846.csv : 対象列が存在しません。
【スキップ】 center_internal_degree_20261105_160846.csv : 対象列が存在しません。
【スキップ】 periphery_single_20261105_160846.csv : 対象列が存在しません。
【スキップ】 bridge_u_only_20261105_160846.csv : 対象列が存在しません。


In [2]:
import glob
import os
import pandas as pd

# --------------------------------------------------
# 変更したい2つのカラム名をペアで指定
# --------------------------------------------------
column_mapping = {
    "true_only": "indep_with_false",  # 1つ目の列 (旧名 : 新名)
    "true_with_false": "comp_with_false",  # 2つ目の列 (旧名 : 新名)
}

# フォルダパスを入力して実行
target_folder = input(
    "../data/LFR01/result/f_com1_user_2"
)
target_folder = target_folder.strip().strip("'\"")

if not os.path.exists(target_folder):
    print(f"エラー: 指定されたフォルダ '{target_folder}' が存在しません。")
else:
    csv_files = glob.glob(os.path.join(target_folder, "*.csv"))

    if not csv_files:
        print(f"フォルダ内にCSVファイルが見つかりませんでした。")
    else:
        print(f"合計 {len(csv_files)} 件のCSVファイルを処理します...\n")

        for file_path in csv_files:
            try:
                df = pd.read_csv(file_path)

                # 対象となる列名がファイル内に含まれているか判定
                matched_cols = [
                    c for c in column_mapping.keys() if c in df.columns
                ]

                if matched_cols:
                    # 2つの列名を一括変更
                    df = df.rename(columns=column_mapping)
                    df.to_csv(file_path, index=False)
                    print(
                        f"【更新成功】 {os.path.basename(file_path)} （変更された列: {matched_cols}）"
                    )
                else:
                    print(
                        f"【スキップ】 {os.path.basename(file_path)} : 対象のカラムが存在しません。"
                    )

            except Exception as e:
                print(
                    f"【エラー】 {os.path.basename(file_path)} の処理中にエラーが発生しました: {e}"
                )

        print("\nすべてのファイルの処理が完了しました。")

AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)